<img src="https://upload.wikimedia.org/wikipedia/commons/3/35/Uba_fiuba_ingenieria_logo.png" width="300" align="center">



# **Analisis de Series de Tiempo II**

# **Tarea 2, Global Forecasting Multivariado**

## Enunciado completo


**Límite de entrega para ser evaluado sobre 100 puntos:** 12 de Agosto, 23:59 (UTC-3)

Posteriormente pueden enviar la solución de la tarea, pero cada semana de atraso reduce 10 puntos la evaluación.

**Parte 1) Preparación de datos, 20 puntos**

* Fija las semillas de Python, NumPy y del
framework de deep learning que utilices.

* El archivo contiene una única serie temporal. Conviértela a formato panel generando entre 4 y 12 series, cada una identificada
por un series_id. Justifica el criterio de segmentación elegido. Ten en cuenta que en la Parte 2 deberás entrenar un modelo por serie: el número de series que elijas determina tu costo computacional.
* Horizonte de 24 horas.
* Variable objetivo: pollution (PM2.5).
* Longitud de la ventana de entrada y justifica la elección en términos de la dinámica de la serie.

* Define el tamaño del test y justifica el tamaño elegido.

* Construye los tensores de entrada y salida mediante ventanas deslizantes.

**Parte 2) Modelos, 40 puntos**

Todas las configuraciones de esta parte se evalúan sobre el mismo conjunto de test.

* Implementa un Baseline. Justifica tu elección y explica qué significaría que una red neuronal no lograra superarlo.

* Implementa un Modelo A. Una RNN simple compuesta por una capa recurrente (LSTM o GRU) seguida de la capa de salida. Documenta la arquitectura y el número de parámetros.

* Implementa un Modelo B. Una RNN profunda, arquitectura recurrente multicapa (utilizando LSTM y/o GRU). Documenta arquitectura y parámetros.

Entrena ahora las cuatro configuraciones siguientes:

| Configuración | Descripción |
|---|---|
| **Local, Modelo A** | Un modelo independiente por cada serie del panel |
| **Global, Modelo A** | Un único modelo entrenado sobre todas las series |
| **Local, Modelo B** | Un modelo independiente por cada serie del panel |
| **Global, Modelo B** | Un único modelo entrenado sobre todas las series |


**Parte 3) Embedding categórico, 20 puntos**

Incorpora series_id como variable categórica mediante una capa de *embedding*, enlas dos configuraciones globales.


Debes justificar:
* la dimensión del embedding elegida
* cómo se integra el vector estático con la secuencia de entrada

**Parte 4) Comparativa e interpretación, 10 puntos**

Tabla comparativa. Reúne las siete configuraciones: el baseline, las
cuatro de la Parte 2 y las dos de la Parte 3 en una única tabla que incluya:

- MAE
- número total de parámetros
- número de modelos entrenados (1 en las globales, N en las locales)
- mejora porcentual respecto al baseline

Acompaña la tabla con al menos un gráfico.

* Redacta la lectura de los resultados. ¿Qué diferencias son
sostenibles a partir de la evidencia disponible y cuáles no?

**Parte 5) Preguntas de análisis, 10 puntos**

* Un modelo global impone una única función a series con dinámicas distintas, lo que parece una desventaja frente a modelos locales especializados. ¿Por qué, aun así, suele obtener mejores resultados? ¿Bajo qué condiciones esperarías lo contrario?

* Supón que observas una mejora del 3% en MAE al añadir el embedding. ¿Qué evidencia adicional necesitas para sostener que esa mejora es real y no producto del azar en la inicialización? Indica qué calcularías concretamente.

**Entregable: notebook con el código corriendo y las respuestas a las 2 preguntas.**

**Link de entrega: [https://forms.gle/AJLpFTwypFJvjYETA](https://forms.gle/AJLpFTwypFJvjYETA)**

## Resolucion

Alumno: Gustavo Julian Rivas -  N° SIU: a1620.

La resolucion se presenta siguiendo cada parte solicitada en el enunciado.


## Parte 1) Preparación de datos

Primero se prepara el entorno, se carga la serie horaria de Beijing y se la convierte al formato panel solicitado.

Se generan **5 series**, una por cada año completo entre 2010 y 2014. El criterio mantiene la frecuencia horaria y el valor original de la variable objetivo: cada serie representa el mismo fenómeno medido durante un año distinto. Esto permite que los modelos globales aprendan patrones compartidos entre años y que los modelos locales se especialicen en cada período, sin crear una cantidad excesiva de modelos.

El horizonte es de **24 horas**: a partir de una ventana que termina en la hora t se predice directamente pollution en t+24. La ventana de entrada tiene **168 horas**, es decir, una semana completa. Esta longitud permite representar persistencia, ciclos diarios y el ciclo semanal observado en la contaminación.

Los últimos **30 días (720 horas) de cada serie** se reservan para test. El tamaño cubre suficientes ciclos diarios y semanales, produce 3600 errores fuera de muestra y deja más de diez meses por serie para entrenamiento. La separación es temporal y se realiza antes de evaluar los modelos.

In [ ]:
# Si alguna dependencia no estuviera disponible en Colab, descomentar la linea siguiente.
%pip install -q tensorflow pandas numpy matplotlib seaborn scikit-learn

import os
import random
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pollution.csv"
YEARS = [2010, 2011, 2012, 2013, 2014]
N_SERIES = len(YEARS)
WINDOW = 168          # una semana de historia horaria
HORIZON = 24          # prediccion directa 24 horas hacia adelante
N_TEST = 30 * 24      # ultimos 30 dias de cada serie
EMBED_DIM = 3
EPOCHS = 60
BATCH_SIZE = 64
FEATURES = ["pollution", "dew", "temp", "press", "wnd_spd"]

plt.style.use("seaborn-v0_8-whitegrid")
print("TensorFlow:", tf.__version__)

### Carga de datos y construcción del panel

Se conserva la variable objetivo pollution y se incorporan como covariables el punto de rocío, la temperatura, la presión y la velocidad acumulada del viento. Los faltantes de PM2.5 se completan solamente con el último valor pasado disponible. Las observaciones iniciales que todavía no tienen historia se eliminan, evitando usar valores futuros para imputar el pasado.

El panel usa `series_id` para identificar el año y `ds` para conservar la fecha horaria real. No se agregan ni promedian observaciones: la variable objetivo sigue siendo el PM2.5 horario original.

In [ ]:
raw = pd.read_csv(URL)
raw["ds"] = pd.to_datetime(raw[["year", "month", "day", "hour"]])
raw = raw.rename(columns={
    "pm2.5": "pollution",
    "DEWP": "dew",
    "TEMP": "temp",
    "PRES": "press",
    "Iws": "wnd_spd",
})
raw = raw.set_index("ds").sort_index()
raw["pollution"] = raw["pollution"].ffill()
raw = raw[raw.index.year.isin(YEARS)].copy()
raw = raw.dropna(subset=FEATURES)
raw["series_id"] = raw.index.year.map(lambda year: f"anio_{year}")

panel = (raw.reset_index()[["series_id", "ds"] + FEATURES]
            .sort_values(["series_id", "ds"])
            .reset_index(drop=True))

assert panel[FEATURES].notna().all().all()
assert panel.groupby("series_id")["ds"].apply(
    lambda s: s.diff().dropna().eq(pd.Timedelta(hours=1)).all()
).all()

resumen_panel = panel.groupby("series_id").agg(
    inicio=("ds", "min"),
    fin=("ds", "max"),
    observaciones=("ds", "size"),
)
display(resumen_panel)
display(panel.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
for sid, g in panel.groupby("series_id"):
    axes[0].plot(g["ds"], g["pollution"], lw=.5, alpha=.8, label=sid)

ejemplo = panel[panel["series_id"] == "anio_2010"].tail(14 * 24)
axes[1].plot(ejemplo["ds"], ejemplo["pollution"], lw=1, color="tab:blue")
axes[0].set_title("Panel horario de PM2.5 por año")
axes[0].legend(ncol=5, fontsize=8)
axes[1].set_title("Últimas dos semanas de la primera serie")
axes[1].set_xlabel("Fecha")
for ax in axes:
    ax.set_ylabel("PM2.5")
plt.tight_layout()
plt.show()

### Construcción de ventanas deslizantes

Cada entrada tiene forma (168, 5): 168 horas y cinco variables. El objetivo es pollution 24 horas después de la última observación de la ventana.

Cada variable se estandariza con media y desvío calculados exclusivamente sobre el tramo de entrenamiento de su serie. Para el test se permiten ventanas que usan el final del entrenamiento, porque esa historia ya estaría disponible al momento de pronosticar; el objetivo siempre pertenece al período de test.

In [ ]:
series_ids = sorted(panel["series_id"].unique())
id_map = {sid: i for i, sid in enumerate(series_ids)}
X_train, y_train, X_test, y_test, scales = {}, {}, {}, {}, {}

for sid in series_ids:
    g = panel[panel["series_id"] == sid].sort_values("ds").reset_index(drop=True)
    values = g[FEATURES].to_numpy(dtype="float32")
    train_end = len(g) - N_TEST

    mu = values[:train_end].mean(axis=0)
    sd = values[:train_end].std(axis=0)
    sd[sd == 0] = 1.0
    scales[sid] = {"mu": mu, "sd": sd, "dates": g["ds"].to_numpy()}
    z = (values - mu) / sd

    Xs, ys, target_pos = [], [], []
    for t in range(WINDOW, len(z) - HORIZON + 1):
        target_t = t + HORIZON - 1
        Xs.append(z[t-WINDOW:t])
        ys.append(z[target_t, 0])
        target_pos.append(target_t)

    Xs = np.asarray(Xs, dtype="float32")
    ys = np.asarray(ys, dtype="float32").reshape(-1, 1)
    target_pos = np.asarray(target_pos)
    is_test = target_pos >= train_end

    X_train[sid], y_train[sid] = Xs[~is_test], ys[~is_test]
    X_test[sid], y_test[sid] = Xs[is_test], ys[is_test]
    assert len(y_test[sid]) == N_TEST

print("Series:", id_map)
for sid in series_ids:
    print(sid, "train:", X_train[sid].shape, "test:", X_test[sid].shape)

## Parte 2) Modelos

### Baseline

Se utiliza un **naive estacional diario**. Como el objetivo está 24 horas por delante de la última observación de entrada, el baseline predice que pollution será igual a ese último valor observado, correspondiente a la misma hora del día anterior.

Es un baseline apropiado porque la contaminación presenta persistencia y ciclo diario. Si una red neuronal no logra superarlo, su complejidad no queda justificada: el modelo no estaría extrayendo información estable adicional de la semana de historia y de las covariables.

### Arquitecturas recurrentes

- **Modelo A:** una GRU de 24 unidades seguida por una capa Dense de una salida.
- **Modelo B:** una LSTM de 32 unidades con `return_sequences=True`, dropout de 0.10, una GRU de 16 unidades y una capa Dense de una salida.

Ambos producen un pronóstico directo para t+24. Se usa la misma configuración de optimización en los enfoques locales y globales para que la comparación dependa principalmente del pooling y de la arquitectura.

In [ ]:
def reset_seed(offset=0):
    keras.backend.clear_session()
    random.seed(SEED + offset)
    np.random.seed(SEED + offset)
    keras.utils.set_random_seed(SEED + offset)

def build_model_a(n_features=len(FEATURES), with_embedding=False):
    seq = keras.Input((WINDOW, n_features), name="secuencia")
    if with_embedding:
        sid = keras.Input((1,), dtype="int32", name="series_id")
        emb = layers.Embedding(N_SERIES, EMBED_DIM, name="embedding_series")(sid)
        emb = layers.RepeatVector(WINDOW)(layers.Flatten()(emb))
        x = layers.Concatenate(axis=-1)([seq, emb])
        inputs = [seq, sid]
    else:
        x, inputs = seq, seq
    x = layers.GRU(24, name="gru")(x)
    out = layers.Dense(1, name="pollution_24h")(x)
    return keras.Model(inputs, out, name="modelo_A" + ("_embedding" if with_embedding else ""))

def build_model_b(n_features=len(FEATURES), with_embedding=False):
    seq = keras.Input((WINDOW, n_features), name="secuencia")
    if with_embedding:
        sid = keras.Input((1,), dtype="int32", name="series_id")
        emb = layers.Embedding(N_SERIES, EMBED_DIM, name="embedding_series")(sid)
        emb = layers.RepeatVector(WINDOW)(layers.Flatten()(emb))
        x = layers.Concatenate(axis=-1)([seq, emb])
        inputs = [seq, sid]
    else:
        x, inputs = seq, seq
    x = layers.LSTM(32, return_sequences=True, name="lstm_1")(x)
    x = layers.Dropout(.10)(x)
    x = layers.GRU(16, name="gru_2")(x)
    out = layers.Dense(1, name="pollution_24h")(x)
    return keras.Model(inputs, out, name="modelo_B" + ("_embedding" if with_embedding else ""))

for builder in (build_model_a, build_model_b):
    m = builder()
    print(m.name, f"{m.count_params():,} parámetros")
    m.summary()

In [ ]:
def compile_and_fit(model, x, y, validation_data=None):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"],
    )
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
    )
    fit_kwargs = dict(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stopping],
        verbose=0,
        shuffle=True,
    )
    if validation_data is None:
        fit_kwargs["validation_split"] = .15
    else:
        fit_kwargs["validation_data"] = validation_data
    return model.fit(x, y, **fit_kwargs)

def original_values(sid, y_z):
    return np.asarray(y_z).reshape(-1) * scales[sid]["sd"][0] + scales[sid]["mu"][0]

def pooled_mae(y_true_by_series, pred_by_series):
    errors = []
    for sid in series_ids:
        y_true = original_values(sid, y_true_by_series[sid])
        errors.extend(np.abs(y_true - np.asarray(pred_by_series[sid]).reshape(-1)))
    return float(np.mean(errors))

# La última observación de cada ventana está exactamente 24 horas antes del target.
baseline_pred = {
    sid: original_values(sid, X_test[sid][:, -1, 0])
    for sid in series_ids
}
baseline_mae = pooled_mae(y_test, baseline_pred)
print(f"Baseline naive estacional diario - MAE: {baseline_mae:.3f}")

In [ ]:
def train_local(builder, seed_offset=0):
    preds, models, histories = {}, {}, {}
    for j, sid in enumerate(series_ids):
        reset_seed(seed_offset + j)
        model = builder()
        histories[sid] = compile_and_fit(model, X_train[sid], y_train[sid])
        pred_z = model.predict(X_test[sid], verbose=0).reshape(-1)
        preds[sid] = original_values(sid, pred_z)
        models[sid] = model
    return preds, models, histories

local_a_pred, local_a_models, hist_local_a = train_local(build_model_a, 100)
local_b_pred, local_b_models, hist_local_b = train_local(build_model_b, 200)
print("MAE Local A:", pooled_mae(y_test, local_a_pred))
print("MAE Local B:", pooled_mae(y_test, local_b_pred))

In [ ]:
# Para el modelo global se separa primero el último 15% temporal de cada serie.
# Luego se construyen pools independientes de entrenamiento y validación.
train_parts, val_parts = [], []
y_train_parts, y_val_parts = [], []
id_train_parts, id_val_parts = [], []

for sid in series_ids:
    n_val = max(1, int(len(X_train[sid]) * .15))
    train_parts.append(X_train[sid][:-n_val])
    val_parts.append(X_train[sid][-n_val:])
    y_train_parts.append(y_train[sid][:-n_val])
    y_val_parts.append(y_train[sid][-n_val:])
    id_train_parts.append(np.full(len(X_train[sid]) - n_val, id_map[sid], dtype="int32"))
    id_val_parts.append(np.full(n_val, id_map[sid], dtype="int32"))

X_pool = np.concatenate(train_parts)
y_pool = np.concatenate(y_train_parts)
id_pool = np.concatenate(id_train_parts)
X_val_pool = np.concatenate(val_parts)
y_val_pool = np.concatenate(y_val_parts)
id_val_pool = np.concatenate(id_val_parts)

def train_global(builder, with_embedding=False, seed_offset=0):
    reset_seed(seed_offset)
    model = builder(with_embedding=with_embedding)
    if with_embedding:
        x_fit = [X_pool, id_pool.reshape(-1, 1)]
        x_val = [X_val_pool, id_val_pool.reshape(-1, 1)]
    else:
        x_fit = X_pool
        x_val = X_val_pool
    history = compile_and_fit(model, x_fit, y_pool, validation_data=(x_val, y_val_pool))

    preds = {}
    for sid in series_ids:
        ids = np.full((len(X_test[sid]), 1), id_map[sid], dtype="int32")
        x_te = [X_test[sid], ids] if with_embedding else X_test[sid]
        pred_z = model.predict(x_te, verbose=0).reshape(-1)
        preds[sid] = original_values(sid, pred_z)
    return preds, model, history

global_a_pred, global_a_model, hist_global_a = train_global(build_model_a, False, 300)
global_b_pred, global_b_model, hist_global_b = train_global(build_model_b, False, 400)
print("MAE Global A:", pooled_mae(y_test, global_a_pred))
print("MAE Global B:", pooled_mae(y_test, global_b_pred))

## Parte 3) Embedding categórico

Se incorpora `series_id` en las dos configuraciones globales. Para cinco categorías se elige una dimensión de embedding igual a **3**. Es una representación compacta: permite aprender diferencias y similitudes entre años sin agregar una cantidad innecesaria de parámetros.

El vector es estático para cada serie. Primero se obtiene el embedding correspondiente al `series_id`, luego se lo repite a lo largo de los 168 pasos y finalmente se lo concatena con las cinco variables de cada hora. De esta forma, la red recibe en cada paso tanto la información multivariada como la identidad de la serie.

In [ ]:
global_a_emb_pred, global_a_emb_model, hist_global_a_emb = train_global(build_model_a, True, 500)
global_b_emb_pred, global_b_emb_model, hist_global_b_emb = train_global(build_model_b, True, 600)
print("MAE Global A + embedding:", pooled_mae(y_test, global_a_emb_pred))
print("MAE Global B + embedding:", pooled_mae(y_test, global_b_emb_pred))
global_a_emb_model.summary()
global_b_emb_model.summary()

## Parte 4) Comparativa e interpretación

La tabla reúne las siete configuraciones pedidas. El MAE se calcula en la escala original de PM2.5 agrupando las predicciones de las cinco series, de manera que cada observación de test tenga el mismo peso.

En los enfoques locales se informa la suma de los parámetros de los cinco modelos que deben mantenerse. En los globales se informa la cantidad de parámetros del único modelo.

In [ ]:
configs = [
    ("Baseline seasonal naive", baseline_mae, 0, 0),
    ("Local, Modelo A", pooled_mae(y_test, local_a_pred), sum(m.count_params() for m in local_a_models.values()), N_SERIES),
    ("Global, Modelo A", pooled_mae(y_test, global_a_pred), global_a_model.count_params(), 1),
    ("Local, Modelo B", pooled_mae(y_test, local_b_pred), sum(m.count_params() for m in local_b_models.values()), N_SERIES),
    ("Global, Modelo B", pooled_mae(y_test, global_b_pred), global_b_model.count_params(), 1),
    ("Global, Modelo A + embedding", pooled_mae(y_test, global_a_emb_pred), global_a_emb_model.count_params(), 1),
    ("Global, Modelo B + embedding", pooled_mae(y_test, global_b_emb_pred), global_b_emb_model.count_params(), 1),
]
results = pd.DataFrame(configs, columns=["configuracion", "MAE", "parametros_totales", "modelos_entrenados"])
results["mejora_vs_baseline_pct"] = 100 * (baseline_mae - results["MAE"]) / baseline_mae
results = results.sort_values("MAE").reset_index(drop=True)
display(results.style.format({"MAE": "{:.3f}", "mejora_vs_baseline_pct": "{:.2f}%", "parametros_totales": "{:,.0f}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_df = results.sort_values("MAE", ascending=False)
sns.barplot(data=plot_df, x="MAE", y="configuracion", ax=axes[0], color="steelblue")
axes[0].axvline(baseline_mae, color="firebrick", ls="--", label="Baseline")
axes[0].set_title("MAE global en test (menor es mejor)"); axes[0].legend()
sns.scatterplot(data=results, x="parametros_totales", y="MAE", hue="modelos_entrenados",
                size="modelos_entrenados", sizes=(80, 180), ax=axes[1], palette="viridis")
for _, r in results.iterrows():
    axes[1].annotate(r["configuracion"].replace("Modelo ", "M"), (r["parametros_totales"], r["MAE"]), fontsize=7)
axes[1].set_xscale("symlog"); axes[1].set_title("Precisión y costo de mantenimiento")
plt.tight_layout(); plt.show()

In [ ]:
all_preds = {
    "Baseline": baseline_pred, "Local A": local_a_pred, "Global A": global_a_pred,
    "Local B": local_b_pred, "Global B": global_b_pred,
    "Global A + emb": global_a_emb_pred, "Global B + emb": global_b_emb_pred,
}
per_series = []
for name, preds in all_preds.items():
    for sid in series_ids:
        yt = original_values(sid, y_test[sid])
        per_series.append({"configuracion": name, "series_id": sid,
                           "MAE": np.mean(np.abs(yt - preds[sid]))})
per_series = pd.DataFrame(per_series)
display(per_series.pivot(index="series_id", columns="configuracion", values="MAE").round(2))

winner = results.iloc[0]
print(f"Mejor configuración puntual: {winner.configuracion}, MAE={winner.MAE:.3f}.")
print("La tabla por serie permite verificar si la ventaja es general o depende de una franja.")

### Lectura de los resultados

La tabla permite identificar qué configuración obtiene el menor MAE en este test temporal y cuánto mejora respecto del naive diario. El desglose por serie permite verificar si una ventaja se repite entre años o si depende de un período particular. También permite comparar precisión y costo: un modelo global mantiene una sola red, mientras que cada enfoque local requiere cinco.

Lo que puede sostenerse con esta evidencia es una comparación descriptiva sobre el mismo conjunto de test. Si una configuración mejora el MAE agregado y además gana en la mayoría de los años, la evidencia es más consistente que si toda la mejora proviene de una única serie.

No puede afirmarse que una arquitectura, el enfoque global o el embedding sean superiores en general usando una sola semilla y un solo holdout. Esas conclusiones requieren repetir el entrenamiento y evaluar varios orígenes temporales.

## Parte 5) Preguntas de análisis

### Pregunta 1: modelo global frente a modelos locales

Un modelo global entrena con las ventanas de todas las series. Aunque comparte una única función, dispone de una muestra mucho mayor que cada modelo local. Esto reduce la varianza, actúa como regularización y permite transferir patrones comunes, como persistencia, ciclos diarios y relaciones entre contaminación y variables meteorológicas. La estandarización por serie evita que un año domine solamente por su escala, y el embedding permite incorporar cierta especialización sin perder el pooling.

Esperaría que los modelos locales fueran mejores si las series tuvieran dinámicas muy diferentes o relaciones contradictorias, si hubiera poca estructura compartida, si una serie dominara el pool, si el modelo global tuviera capacidad insuficiente o si cada serie local dispusiera de muchos datos estables. En esos casos puede aparecer transferencia negativa.

### Pregunta 2: evidencia necesaria para una mejora de 3% por embedding

Entrenaría los modelos globales con y sin embedding usando, por ejemplo, 20 semillas emparejadas. Mantendría los mismos splits, arquitectura, épocas máximas y criterio de early stopping. Para cada semilla calcularía la diferencia `MAE_sin_embedding - MAE_con_embedding`.

Sobre esas diferencias calcularía la media, el desvío, la proporción de semillas en las que gana el embedding y un intervalo de confianza del 95% mediante bootstrap emparejado. También repetiría la comparación en varios orígenes temporales con rolling origin y reportaría resultados por origen y por serie.

La mejora sería más creíble si el intervalo de confianza de la diferencia quedara por encima de cero, apareciera en la mayoría de las semillas y orígenes, y no dependiera de un único año. Un test emparejado de permutación o Wilcoxon puede complementar el intervalo, pero no reemplaza el análisis del tamaño y la estabilidad del efecto.

## Conclusiones

La resolución conserva la frecuencia horaria de la serie original y formula de manera explícita un pronóstico directo de pollution a 24 horas. El panel de cinco años permite comparar modelos locales y globales con un costo computacional razonable.

Todas las configuraciones usan el mismo test temporal y el escalado se calcula solamente con datos de entrenamiento. El naive diario funciona como referencia fuerte. Los modelos globales aprovechan información compartida, mientras que los embeddings agregan la identidad de la serie. La tabla final permite evaluar conjuntamente precisión, mejora frente al baseline y costo de mantenimiento.